In [95]:
import numpy as np
import networkx as nx

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate, CXGate, QAOAAnsatz

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qopt_best_practices.sat_mapping import SATMapper
from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper

from qiskit.quantum_info import SparsePauliOp


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import HighLevelSynthesis, InverseCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping


from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti

In [96]:
extended_swap_strat = ExtendedSwapStrategy.from_heavy_hex(2, 2)
num_physical_qubits = extended_swap_strat._num_vertices

In [97]:
n = 3
T = 3
num_qubits = n*T

In [98]:
rng = np.random.default_rng()
doubles = rng.choice(num_qubits, (20, 2))
quads = rng.choice(num_qubits, (2, 4), replace=False)
coeffs = rng.random(22)

In [99]:
def choice_to_pauli(choice: list[int], size: int) -> str:
    pauli = ['I'] * size
    for x in choice:
        pauli[x] = 'Z'
    return ''.join(pauli)

In [100]:
hamiltonian = SparsePauliOp([choice_to_pauli(c, num_qubits) for c in doubles] + [choice_to_pauli(c, num_qubits) for c in quads], coeffs=coeffs)
hamiltonian = hamiltonian.simplify()
hamiltonian = hamiltonian.sort(weight=True)


In [7]:
def hamiltonian_to_doubles_graph(hamiltonian: SparsePauliOp) -> nx.Graph:
    edges = []
    weights = []
    for t in hamiltonian:
        if np.sum(t.paulis[0].z) == 2:
            edge = np.nonzero(t.paulis[0].z)[0]
            edges.append(edge)
            weights.append(t.coeffs[0])
            
    program_graph = nx.Graph()
    for i in range(hamiltonian.num_qubits):
        program_graph.add_node(i)
    for idx in range(len(weights)):
        program_graph.add_edge(edges[idx][0],edges[idx][1],weight=weights[idx])
    return program_graph


In [101]:
def hamiltonian_to_interactions(hamiltonian: SparsePauliOp) -> list[tuple]:
    interactions = []
    for t in hamiltonian:
        if np.sum(t.paulis[0].z) < 1 or np.sum(t.paulis[0].z) > 5:
            continue
        if np.sum(t.paulis[0].z) == 2:
            edge = np.nonzero(t.paulis[0].z)[0]
            interactions.append(edge)
        if np.sum(t.paulis[0].z) == 4:
            edge = np.nonzero(t.paulis[0].z)[0]
            interactions.append(edge)
    return interactions

In [102]:
def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')

In [103]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id"]
config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(AerSimulator._DEFAULT_CONFIGURATION)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map)

In [ ]:
backend._basis_gates()

In [ ]:
pm = PassManager(
    [
        HighLevelSynthesis(basis_gates=["PauliEvolution"]), # Not needed if set up circuit as PauliEvolutionGate
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            extended_swap_strat,
        ),
        SwapToFinalMapping(),
        HighLevelSynthesis(basis_gates=["sx", "x", "rz", "rzz", "cx", "id"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [ ]:
program_graph = hamiltonian_to_doubles_graph(hamiltonian)


In [ ]:
sat_results = SATMapper(timeout=60).find_initial_mappings(
    program_graph, extended_swap_strat, 0, len(extended_swap_strat)
)
solutions = [k for k, v in sat_results.items() if v.satisfiable]


In [ ]:
solutions

In [ ]:
if len(solutions):
    min_k = min(solutions)
    print(f'Min SWAP layers to satisfy doubles: {min_k}')
    edge_map = dict(sat_results[min_k].mapping)
    print(f'Doubles edge map: {edge_map}')

    new_hamiltonian = hamiltonian.apply_layout([edge_map[i] for i in range(num_qubits)], num_physical_qubits)
    
    new_cost_circ = QuantumCircuit(num_physical_qubits)
    new_cost_circ.append(PauliEvolutionGate(new_hamiltonian), range(num_physical_qubits))
    new_tcost = pm.run(new_cost_circ)
    
    print_circuit_info(new_tcost, 'Remapped, commuting gate routed circuit')
    print(new_tcost.count_ops())
    
    backend_new_tcost = transpile(new_tcost, optimization_level=3, backend=backend, basis_gates=basis_gates)
    
    print_circuit_info(backend_new_tcost, 'Remapped, commuting gate routed circuit on backend')
    print(backend_new_tcost.count_ops())

In [105]:
qaoa_cost_op = QAOAAnsatz(
    hamiltonian,
    mixer_operator=QuantumCircuit(num_qubits),
    initial_state=QuantumCircuit(num_qubits)
)
backend_tqaoa = transpile(qaoa_cost_op, optimization_level=3, backend=backend, basis_gates=basis_gates)

print_circuit_info(backend_tqaoa, 'Default qaoa circuit on backend')
print(backend_tqaoa.count_ops())

Default qaoa circuit on backend has 77 2Q gates     and 54 2Q depth
OrderedDict([('sx', 122), ('cz', 63), ('rz', 36), ('rzz', 14)])


/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [106]:
program_interactions = hamiltonian_to_interactions(hamiltonian)
nodes = set([node for interaction in program_interactions for node in interaction])
nodes

{np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8)}

In [107]:
sat_results = HigherOrderSatMapper(timeout=5).find_hubo_mappings(
    program_interactions, extended_swap_strat, 0, len(extended_swap_strat)
)
solutions = [k for k, v in sat_results.items() if v.satisfiable]

9 35
Num layers: 50
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 25
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 12
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 6
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 3
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 5
Start populating order 4 distance tensor
Finished populating order 4 distance tensor
Num layers: 4
Start populating order 4 distance tensor
Finished populating order 4 distance tensor


In [108]:
solutions

[50, 25, 12, 6, 5, 4]

In [133]:
if len(solutions):
    min_k = min(solutions)
    print(f'Min SWAP layers to satisfy HUBO: {min_k}')
    edge_map = dict(sat_results[min_k].mapping)
    print(f'Doubles edge map: {edge_map}')

    new_hamiltonian = hamiltonian.apply_layout([edge_map[i] for i in range(num_qubits)], num_physical_qubits)
    
    new_cost_circ = QuantumCircuit(num_physical_qubits)
    new_cost_circ.append(PauliEvolutionGate(new_hamiltonian), range(num_physical_qubits))
    new_tcost = pm.run(new_cost_circ)
    
    print_circuit_info(new_tcost, 'Remapped, commuting gate routed circuit')
    print(new_tcost.count_ops())
    
    backend_new_tcost = transpile(new_tcost, optimization_level=3, backend=backend, basis_gates=basis_gates)
    
    print_circuit_info(backend_new_tcost, 'Remapped, commuting gate routed circuit on backend')
    print(backend_new_tcost.count_ops())

Min SWAP layers to satisfy HUBO: 4
Doubles edge map: {0: 22, 1: 2, 2: 9, 3: 27, 4: 20, 5: 4, 6: 21, 7: 1, 8: 0}
16:03:08 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 0
Remapped, commuting gate routed circuit has 0 2Q gates     and 12 2Q depth
OrderedDict([('swap', 26), ('PauliEvolution', 16)])
Remapped, commuting gate routed circuit on backend has 68 2Q gates     and 33 2Q depth
OrderedDict([('sx', 105), ('cz', 54), ('rz', 30), ('rzz', 14)])


In [134]:
new_tcost.draw(fold=-1)

q_0 -> 0 ─X───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
            │                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           
  q_1 -> 1 ─┼──X────────────────────────────────────────────────────────────────────────────────────────X──────────────────────────────────────────────────────────────────────────────────────────────────X────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
            │  │ ┌───────────────────────┐                                                              │                                                                        ┌───────────────────────┐ │                                   ┌──────────────────────┐                  ┌──────────────────────┐                        ┌───────────────────────┐                              ┌───────────────────────┐                           ┌─────────────────────────┐┌───────────────────────┐
  q_2 -> 2 ─┼──┼─┤0                      ├──────────────────────────────────────────────────────────────┼──X─────────────────────────────────X───────────────────────────────────┤0                      ├─┼───────────────────────────────────┤0                     ├──────────X───────┤1                     ├────────────────────────┤1                      ├────X─────────────────────────┤0                      ├───────────────────────────┤0                        ├┤0                      ├
            │  │ │                       │                                                              │  │                                 │ ┌───────────────────────┐         │                       │ │                                   │                      │          │       │                      │┌──────────────────────┐│                       │    │ ┌──────────────────────┐│                       │┌─────────────────────────┐│                         ││                       │
  q_3 -> 3 ─┼──┼─┤                       ├─X────────────────────────────────────────────────────────────┼──┼─────────────────────────────────┼─┤1                      ├─────────┤                       ├─┼──X────────────────────────────────┤                      ├─X────────┼───────┤                      ├┤1                     ├┤                       ├─X──┼─┤0                     ├┤                       ├┤0                        ├┤                         ├┤                       ├
            │  │ │                       │ │ ┌──────────────────────┐                                   │  │                                 │ │                       │         │                       │ │  │                                │                      │ │        │       │                      ││                      ││                       │ │  │ │                      ││                       ││                         ││                         ││                       │
  q_4 -> 4 ─┼──┼─┤                       ├─┼─┤0                     ├─────────────────────

In [118]:
backend_tqaoa.draw(fold=-1)

global phase: π/2
                                                                                                                                                                                                                                                                                               ┌────┐                                                      ┌────┐         ┌────┐         ┌─────────┐┌────┐          ┌───────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              ┌────┐┌───────┐┌─────────────────────────────┐┌────┐┌───────┐         ┌────┐  ┌─────────┐                                                                                                                                                                                                      
 q_8 -> 1 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────────────────────────────────────────┤ √X ├──────────────────────────────────────────────────■───┤ √X ├─────■───┤ √X ├─────■───┤ Rz(π/2) ├┤ √X ├──────────┤ Rz(π) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─┤ √X ├┤ Rz(π) ├┤ Rz(1.9024710084475187*γ[0]) ├┤ √X ├┤ Rz(π) ├──■──────┤ √X ├──┤ Rz(π/2) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                             ┌────┐                     ┌────┐   ┌────┐   ┌────┐   ┌────┐   ┌────┐    │                                        └────┘                                                  │   └────┘     │   └────┘     │   └─────────┘└────┘          └───────┘                                                                                      ┌────┐                          ┌────┐               ┌────┐                                                                         ┌─────────┐   ┌────┐    ┌───────┐          ┌────┐┌─────────┐         ┌────┐                    ┌────┐   ┌────┐   ┌────┐      ┌────┐      ┌────┐                                         ┌────┐                       ┌────┐                            ┌────┐                    ┌────┐    ┌────┐                 ┌────┐   ┌────┐ │ └────┘└─┬────┬┘└─────────────────────────────┘├────┤└───────┘  │      └────┘  └─────────┘                        ┌─────────┐┌────┐┌──────────┐   ┌────┐┌─────────┐                           ┌────┐   ┌────┐   ┌─────────┐┌────┐┌───────┐      ┌────┐  ┌─────────┐                             
 q_7 -> 2 ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────────────────────────────────────────┤ √X ├───────────────────■─┤ √X ├─■─┤ √X ├─■─┤ √X ├─■─┤ √X ├─■─┤ √X ├─■──┼──────